In [ ]:
RUN_DATE = "2026-10-28"
SOURCE_FILE = "Returns_batch_001.csv"
SOURCE_SYSTEM = "synthetic_returns_generator"

In [ ]:
from pyspark.sql import functions as F

TARGET_TABLE = "bronze.returns_raw"
SOURCE_PATH = (
    f"Files/bronze/returns/landing/"
    f"ingestion_date={RUN_DATE}/{SOURCE_FILE}"
)
BATCH_ID = f"{SOURCE_FILE.removesuffix('.csv').lower()}_{RUN_DATE.replace('-', '')}"

#source contract: generator column -> Bronze column, in file order
COLUMN_MAPPING = {
    "return_id": "return_id",
    "order_number": "order_number",
    "line_item": "line_item",
    "return_date": "return_date_raw",
    "returned_quantity": "returned_quantity_raw",
    "return_reason": "return_reason_raw",
    "item_condition": "item_condition_raw",
    "return_channel": "return_channel_raw",
    "return_store_key": "return_store_key",
    "refund_status": "refund_status_raw",
    "refund_date": "refund_date_raw",
    "refund_amount": "refund_amount_raw",
    "refund_currency_code": "refund_currency_code_raw",
    "status_updated_date": "status_updated_date_raw",
    "source_batch_id": "source_batch_id",
    "generator_version": "generator_version",
    "generator_seed": "generator_seed",
}

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(SOURCE_PATH)
)

assert raw_df.columns == list(COLUMN_MAPPING), (
    f"{SOURCE_FILE}: header {raw_df.columns} does not match {list(COLUMN_MAPPING)}"
)

bronze_df = (
    raw_df.select(*[
        F.col(source_name).alias(target_name)
        for source_name, target_name in COLUMN_MAPPING.items()
    ])
    .withColumn("_source_file", F.lit(SOURCE_FILE))
    .withColumn("_source_path", F.lit(SOURCE_PATH))
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("_batch_id", F.lit(BATCH_ID))
    .withColumn("_ingestion_date", F.to_date(F.lit(RUN_DATE)))
    .withColumn("_ingested_at_utc", F.current_timestamp())
)

source_count = raw_df.count()

In [ ]:
#returns arrive in batches: replace only this batch so reruns never duplicate it
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

if spark.catalog.tableExists(TARGET_TABLE):
    spark.sql(f"DELETE FROM {TARGET_TABLE} WHERE _batch_id = '{BATCH_ID}'")
    bronze_df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
else:
    bronze_df.write.format("delta").saveAsTable(TARGET_TABLE)

print(f"Successfully wrote {BATCH_ID} to {TARGET_TABLE}")

In [ ]:
#validate
batch_df = spark.table(TARGET_TABLE).filter(F.col("_batch_id") == BATCH_ID)
target_count = batch_df.count()

metadata_columns = [c for c in batch_df.columns if c.startswith("_")]
missing_metadata_rows = batch_df.filter(
    " OR ".join(f"{c} IS NULL" for c in metadata_columns)
).count()

#Bronze keeps duplicates as received; Silver quarantines them
duplicate_return_id_groups = (
    batch_df.groupBy("return_id").count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Source rows: {source_count:,}")
print(f"Target rows for {BATCH_ID}: {target_count:,}")
print(f"Missing-metadata rows: {missing_metadata_rows:,}")
print(f"Duplicate return_id groups (kept for Silver): {duplicate_return_id_groups:,}")

assert target_count == source_count
assert missing_metadata_rows == 0

display(
    spark.table(TARGET_TABLE)
    .groupBy("_batch_id", "source_batch_id")
    .count()
    .orderBy("_batch_id")
)